# FVLC replication: Colab runner (DA351 Phase 1)
Replicates Tables 1–3 and Figure 3 of *Faithful Vision-Language Interpretation via Concept Bottleneck Models*
(ICLR 2024) on CIFAR-10, CIFAR-100 and CUB.

**Before you start:** Runtime → Change runtime type → **T4 GPU**.
Everything (cached features + results) is saved to Google Drive, so after a disconnect just re-run
**Step 1** and continue from the step that was interrupted. Finished parts are skipped automatically.

## Step 1: Setup (run every session)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = '/content/drive/MyDrive/fvlc'          # everything persistent lives here
REPO_URL  = ''                                     # e.g. 'https://github.com/<you>/fvlc-replication.git'
ZIP_PATH  = f'{DRIVE_DIR}/fvlc-replication.zip'    # used if REPO_URL is empty
os.makedirs(DRIVE_DIR, exist_ok=True)

if not os.path.exists('/content/fvlc-replication'):
    if REPO_URL:
        !git clone -q $REPO_URL /content/fvlc-replication
    else:
        !unzip -q "$ZIP_PATH" -d /content/
%cd /content/fvlc-replication
!pip install -q -r requirements.txt

os.environ['FVLC_ACT_DIR'] = f'{DRIVE_DIR}/activations'   # cached features (≈1–2 GB per dataset)
RESULTS = f'{DRIVE_DIR}/results'
!nvidia-smi -L

## Step 2: Sanity checks (~2 min)

In [ ]:
!python tests/test_core.py

In [ ]:
!python run_experiments.py --dataset cifar10 --limit_train 1000 --fvlc_epochs 5 --n_iters 50 --std_n_iters 50 --perturbations SEED_ONLY WP1_10 --out /content/smoke

## Step 3: CIFAR-10, Tables 1 + 2 (~1–1.5 h)
The first run extracts features (clean + noisy images, 2 models) and caches them on Drive.

In [ ]:
!python run_experiments.py --dataset cifar10 --out $RESULTS

## Step 4: CIFAR-10, Table 3 ablation (~20–40 min)

In [ ]:
!python run_experiments.py --dataset cifar10 --ablation --out $RESULTS

## Step 5: CIFAR-100, Tables 1 + 2 (~1–1.5 h)

In [ ]:
!python run_experiments.py --dataset cifar100 --out $RESULTS

## Step 6: CUB-200-2011, Tables 1 + 2 (~40 min; 1.1 GB download, re-downloaded per session only until its features are cached)

In [ ]:
# CUB images are only needed until their features are cached on Drive
if not os.path.exists(f"{os.environ['FVLC_ACT_DIR']}/cub/test_resnet18_cub_ip0.001.pt"):
    !bash scripts/download_cub.sh
!python run_experiments.py --dataset cub --out $RESULTS

## Step 7: Tables and figures (ours vs paper)

In [ ]:
!python make_tables.py --results $RESULTS
from IPython.display import Markdown, Image, display
display(Markdown(open(f'{RESULTS}/tables.md').read()))
display(Image(f'{RESULTS}/fig_stability.png'))

In [ ]:
# Figure-3-style qualitative plots (needs the raw images; CUB only if data/CUB exists this session)
for ds in ['cifar10', 'cifar100', 'cub']:
    if os.path.exists(f'{RESULTS}/{ds}/layers/WP1_10+WP2+IP_fvlc.pt') and (ds != 'cub' or os.path.exists('data/CUB')):
        !python visualize.py --dataset $ds --perturbation WP1_10+WP2+IP --index 0 --out $RESULTS
        display(Image(f'{RESULTS}/{ds}/fig3_WP1_10+WP2+IP_0.png'))

## Step 8: Download results (to commit to GitHub as `results/`)

In [ ]:
!cd $DRIVE_DIR && zip -qr results_upload.zip results -x "*/layers/*" "*/vanilla.pt"
from google.colab import files
files.download(f'{DRIVE_DIR}/results_upload.zip')

## Optional: sensitivity checks for the report
How much do the results depend on choices the paper leaves unspecified? (README → "Ambiguities")

In [ ]:
for tag, flags in [('gaussian_init', '--fvlc_init gaussian'), ('per_sample', '--pgd_mode per_sample'),
                   ('rho_concept', '--rho_space concept')]:
    out = f'{RESULTS}_{tag}'
    os.makedirs(f'{out}/cifar10', exist_ok=True)
    !cp {RESULTS}/cifar10/vanilla.pt {out}/cifar10/
    !python run_experiments.py --dataset cifar10 --perturbations SEED_ONLY WP1_10 IP --skip_standard {flags} --out {out}

## Protocol B: SEAT-style evaluation (Tables 2 + 3, second reading of the paper)
Here TCPC/TOPC measure how much the **same trained model** changes under a test-time PGD attack on its input
features. This is how SEAT (Hu et al., AAAI 2023; same lab, same PGD code as the official FVLC repo) evaluates
stability. FVLC is anchored to the LF-CBM of its own concept space (Definition 2).
This reuses the vanilla models and cached features from Steps 3–6, so it is fast (~15–30 min per dataset).
Run the cells one at a time and check the log before starting the next.

In [ ]:
RESULTS_B = f'{DRIVE_DIR}/results_seat'
for ds in ['cifar10', 'cifar100', 'cub']:
    os.makedirs(f'{RESULTS_B}/{ds}', exist_ok=True)
    if os.path.exists(f'{RESULTS}/{ds}/vanilla.pt'):
        !cp {RESULTS}/{ds}/vanilla.pt {RESULTS_B}/{ds}/
SEAT = '--protocol seat --pgd_mode per_sample --ablation --skip_standard'

In [ ]:
!nohup python run_experiments.py --dataset cifar10 $SEAT --out $RESULTS_B > $DRIVE_DIR/seat_cifar10.log 2>&1 &

In [ ]:
!tail -n 15 $DRIVE_DIR/seat_cifar10.log

In [ ]:
!nohup python run_experiments.py --dataset cifar100 $SEAT --out $RESULTS_B > $DRIVE_DIR/seat_cifar100.log 2>&1 &

In [ ]:
!tail -n 15 $DRIVE_DIR/seat_cifar100.log

In [ ]:
!nohup python run_experiments.py --dataset cub $SEAT --out $RESULTS_B > $DRIVE_DIR/seat_cub.log 2>&1 &

In [ ]:
!tail -n 15 $DRIVE_DIR/seat_cub.log

In [ ]:
from IPython.display import Markdown, display
from google.colab import files
!python make_tables.py --results $RESULTS_B
display(Markdown(open(f'{RESULTS_B}/tables.md').read()))
!cd $DRIVE_DIR && zip -qr results_seat_upload.zip results_seat -x "*/layers/*" "*/vanilla.pt"
files.download(f'{DRIVE_DIR}/results_seat_upload.zip')

## Protocol C: paired comparison (reproduces the paper's ~90% stability gain)
Each method is compared with **itself** before vs after the perturbation:
- base: vanilla LF-CBM vs the LF-CBM retrained under the perturbation
- FVLC: FVLC trained on the clean pipeline vs FVLC trained under the perturbation, with the same seed and the
  paper's random initialisation (Algorithm 1, line 2)

Reuses the cached features and vanilla models, so it is fast.

In [ ]:
RESULTS_C = f'{DRIVE_DIR}/results_paired'
for ds in ['cifar10', 'cifar100', 'cub']:
    os.makedirs(f'{RESULTS_C}/{ds}', exist_ok=True)
    if os.path.exists(f'{RESULTS}/{ds}/vanilla.pt'):
        !cp {RESULTS}/{ds}/vanilla.pt {RESULTS_C}/{ds}/
PAIRED = '--protocol paired --fvlc_init gaussian --ablation --skip_standard'

In [ ]:
!nohup python run_experiments.py --dataset cifar10 $PAIRED --out $RESULTS_C > $DRIVE_DIR/paired_cifar10.log 2>&1 &

In [ ]:
!tail -n 15 $DRIVE_DIR/paired_cifar10.log

In [ ]:
!nohup python run_experiments.py --dataset cifar100 $PAIRED --out $RESULTS_C > $DRIVE_DIR/paired_cifar100.log 2>&1 &

In [ ]:
!tail -n 15 $DRIVE_DIR/paired_cifar100.log

In [ ]:
!nohup python run_experiments.py --dataset cub $PAIRED --out $RESULTS_C > $DRIVE_DIR/paired_cub.log 2>&1 &

In [ ]:
!tail -n 15 $DRIVE_DIR/paired_cub.log

In [ ]:
from IPython.display import Markdown, display
from google.colab import files
!python make_tables.py --results $RESULTS_C
display(Markdown(open(f'{RESULTS_C}/tables.md').read()))
!cd $DRIVE_DIR && zip -qr results_paired_upload.zip results_paired -x "*/layers/*" "*/vanilla.pt"
files.download(f'{DRIVE_DIR}/results_paired_upload.zip')